# CS 5326 – PA2: continuation run (UltraFeedback DPO + remaining generations)

The first training run finished data prep, baselines, persona LoRA-DPO, SFT, HH-RLHF DPO, and the SFT generations, then
stopped at UltraFeedback DPO (a JSONL-reader bug, now fixed). This notebook runs **only the remaining stages**, reading the
SFT and HH-DPO checkpoints directly from the first run's output:

| Stage | Uses | Writes |
|---|---|---|
| UltraFeedback DPO (policy cuda:0, fp16 reference cuda:1) | previous `sft/best`, previous UF splits | `ultrafeedback_dpo/` |
| HH-DPO generations (GPU 1, in parallel) | previous `hh_dpo/best` | `generations/hh_dpo_hh_eval.json` |
| UF-DPO generations | new `ultrafeedback_dpo/best` | `generations/ultrafeedback_dpo_alpaca_eval.json` |

**Setup:** GPU T4 x2, Internet On, **Add Input → the first run's notebook output** (the version that failed is fine as
long as it shows `model-alignment/outputs/sft/best/`). Then **Save & Run All**. Expected time ≈ 1.2 h.
Settings are identical to the first run (same DPO / decoding hyperparameters, seed 42), so the outputs can be merged.

In [ ]:
# ============================== configuration ==============================
REPO_URL = "https://github.com/alihasnaink/model-alignment.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/model-alignment"
PREV_OUTPUTS = None        # None = auto-detect the attached first-run outputs (folder containing sft/best/model.safetensors)

SMOKE_TEST = False         # True = tiny check of this notebook (~10 min)
OVERLAP_GPU_WORK = True
RESUME_FROM = None         # not used here: previous checkpoints are read in place, nothing is copied
OUTPUT_MODE = "lean"       # delete UF-DPO weights after its generations are written ("full" keeps them, ~4 GB)

SEED = 42
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
# Must match the first run exactly so HH-DPO and UF-DPO are comparable.
DPO_HP = dict(max_length=512, batch_size=1, gradient_accumulation=32, learning_rate=1e-6, beta=0.1,
              optimizer="adamw", weight_decay=0.0, max_grad_norm=1.0, epochs=1, eval_every=100)
GEN_HP = dict(batch_size=16, max_new_tokens=512, repetition_penalty=1.1, no_repeat_ngram_size=0)

## Setup, versions, and tests (identical to the main notebook)

In [ ]:
import os, subprocess, sys
from pathlib import Path

if not Path(REPO_DIR, "submission_adapter.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Using existing repository: {REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
repo = Path(REPO_DIR).resolve()
os.chdir(repo)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

# Keep Kaggle's CUDA-matched PyTorch: requirements.txt does not list torch, and torchao is removed because
# older Kaggle builds conflict with PEFT.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import datasets, peft, torch, transformers, trl

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("trl / peft  :", trl.__version__, "/", peft.__version__)
print("GPUs        :", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Enable a GPU accelerator."
if not SMOKE_TEST:
    assert torch.cuda.device_count() >= 2, "Select the GPU T4 x2 accelerator (policy on cuda:0, reference on cuda:1)."

# Public adapter smoke tests + our edge-case and end-to-end script tests. Stop here if anything fails.
test_env = {**os.environ, "PYTHONPATH": str(repo), "CUDA_VISIBLE_DEVICES": "0"}  # 1 GPU: no DataParallel in the TRL test
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], check=True, env=test_env)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], check=True, env=test_env)

In [ ]:
# ============================== helpers ==============================
import json, shutil, threading, time, zipfile
from IPython.display import FileLink, Image, Markdown, display

OUT = repo / ("outputs_smoke" if SMOKE_TEST else "outputs")
DATA, GEN, EVAL, LOGS = OUT / "data", OUT / "generations", OUT / "eval", OUT / "logs"
for d in (DATA, GEN, EVAL, LOGS):
    d.mkdir(parents=True, exist_ok=True)
ALPACA_EVAL = repo / "eval_sets/alpaca_eval_300_seed42.json"
HH_EVAL = repo / "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_EVAL = repo / "eval_sets/persona_style_eval.json"

if RESUME_FROM and Path(RESUME_FROM).exists():
    # Copy only files we do not already have, so newer local work is never overwritten.
    for src in Path(RESUME_FROM).rglob("*"):
        dest = OUT / src.relative_to(RESUME_FROM)
        if src.is_file() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
    print("Resumed artifacts from", RESUME_FROM)


def flags(hp):
    return [x for key, value in hp.items() for x in (f"--{key}", value)]


def env_for(gpus):
    env = {**os.environ, "PYTHONPATH": str(repo), "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    if gpus is not None:
        env["CUDA_VISIBLE_DEVICES"] = gpus
    return env


def run(cmd, gpus=None, log_name=None):
    # Run a script in the foreground, streaming output to the cell and to outputs/logs/<log_name>.log.
    cmd = [str(c) for c in cmd]
    log_path = LOGS / f"{log_name or Path(cmd[1]).stem}.log"
    print(f"$ [{'GPU ' + gpus if gpus else 'all GPUs'}] {' '.join(cmd)}", flush=True)
    start = time.time()
    with log_path.open("a") as log, subprocess.Popen(cmd, cwd=repo, env=env_for(gpus), stdout=subprocess.PIPE,
                                                       stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}); see {log_path}")
    print(f"   finished in {(time.time() - start) / 60:.1f} min", flush=True)


class Background:
    # Runs a list of commands sequentially on one GPU in a thread; output goes to a log file only.
    def __init__(self, name, commands, gpus):
        self.name, self.commands, self.gpus, self.error = name, commands, gpus, None
        self.log = LOGS / f"background_{name}.log"
        if OVERLAP_GPU_WORK:
            self.thread = threading.Thread(target=self._work, daemon=True)
            self.thread.start()
            print(f"Started background track '{name}' on GPU {gpus} ({len(commands)} commands) -> {self.log}")
        else:
            self.thread = None
            for cmd in commands:
                run(cmd, gpus=None, log_name=f"background_{name}")

    def _work(self):
        try:
            with self.log.open("a") as log:
                for cmd in self.commands:
                    cmd = [str(c) for c in cmd]
                    log.write(f"\n$ {' '.join(cmd)}\n"); log.flush()
                    start = time.time()
                    code = subprocess.run(cmd, cwd=repo, env=env_for(self.gpus), stdout=log, stderr=subprocess.STDOUT).returncode
                    log.write(f"   exit {code} after {(time.time() - start) / 60:.1f} min\n"); log.flush()
                    if code != 0:
                        raise RuntimeError(f"{cmd[1]} exited with {code}")
        except Exception as error:
            self.error = error

    def wait(self):
        if self.thread is None:
            return
        print(f"Waiting for background track '{self.name}'...", flush=True)
        self.thread.join()
        print("".join(self.log.read_text().splitlines(keepends=True)[-25:]))
        if self.error:
            raise RuntimeError(f"background track '{self.name}' failed: {self.error}. Full log: {self.log}")
        print(f"Background track '{self.name}' finished OK.")


def done(stage_dir):
    return (Path(stage_dir) / "DONE").exists()


KEEP_IN_PRUNED = {"config.json", "generation_config.json", "checkpoint_info.json", "PRUNED.txt"}

def prune_weights(*checkpoint_dirs):
    # Lean mode: drop weights/tokenizer files but keep config + checkpoint_info.json as a record of what was saved.
    if OUTPUT_MODE != "lean":
        return
    for ckpt in map(Path, checkpoint_dirs):
        if not ckpt.exists() or (ckpt / "PRUNED.txt").exists():
            continue
        freed = 0
        for item in ckpt.iterdir():
            if item.name in KEEP_IN_PRUNED:
                continue
            freed += sum(f.stat().st_size for f in item.rglob("*") if f.is_file()) if item.is_dir() else item.stat().st_size
            shutil.rmtree(item) if item.is_dir() else item.unlink()
        (ckpt / "PRUNED.txt").write_text("Weights removed (OUTPUT_MODE='lean') after all generations using them finished.\n")
        print(f"pruned {ckpt.relative_to(OUT)}  (freed {freed / 1e9:.2f} GB)")


def has_weights(ckpt):
    return any(Path(ckpt).glob("*.safetensors"))


def gen_cmd(model, inputs, output, generator, dataset, extra=()):
    return ["python", "scripts/generate.py", "--model", model, "--inputs", inputs, "--output", output,
            "--generator", generator, "--dataset", dataset, *flags(GEN_HP), *SMOKE_GEN, *extra]


# Tiny sizes for SMOKE_TEST; empty for the real run.
SMOKE_PREP = ["--train_size", 200, "--validation_size", 50] if SMOKE_TEST else []
SMOKE_SFT = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_DPO = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_GEN = ["--limit", 6, "--max_new_tokens", 64] if SMOKE_TEST else []
SMOKE_PERSONA = ["--max_steps", 4] if SMOKE_TEST else []
print("Outputs ->", OUT)

## Locate the first run's outputs

In [ ]:
if PREV_OUTPUTS is None:
    hits = sorted(Path("/kaggle/input").rglob("sft/best/model.safetensors"))
    assert hits, ("Attach the first run's notebook output: no sft/best/model.safetensors found under /kaggle/input. "
                  "Use Add Input -> Your Work -> Notebooks -> kaggle-orchestrator (pick the failed version).")
    PREV = hits[0].parent.parent.parent
else:
    PREV = Path(PREV_OUTPUTS)
PREV_SFT_BEST, PREV_HH_BEST = PREV / "sft" / "best", PREV / "hh_dpo" / "best"
for ckpt in (PREV_SFT_BEST, PREV_HH_BEST):
    assert (ckpt / "model.safetensors").exists(), f"missing weights in {ckpt}"
    print(ckpt, json.loads((ckpt / "checkpoint_info.json").read_text()))
assert done(PREV / "sft") and done(PREV / "hh_dpo"), "first run's SFT / HH-DPO are not marked DONE"
print("Previous outputs:", PREV)

In [ ]:
# Reuse the exact UltraFeedback splits from the first run (same files the HH run's notebook prepared).
UF_DIR = DATA / "ultrafeedback"
prev_uf = PREV / "data" / "ultrafeedback"
if (prev_uf / "ultrafeedback_valid.jsonl").exists():
    shutil.copytree(prev_uf, UF_DIR, dirs_exist_ok=True)
    print("Using first-run UltraFeedback splits")
elif not (UF_DIR / "ultrafeedback_valid.jsonl").exists():
    # Deterministic fallback (seed 42): produces the same splits as the first run.
    run(["python", "scripts/prepare_preferences.py", "--track", "ultrafeedback", "--output_dir", UF_DIR, "--seed", SEED, *SMOKE_PREP])
from alignment.training import load_json_records  # fixed reader: splits on "\n" only
for split in ("train", "valid"):
    print(split, len(load_json_records(UF_DIR / f"ultrafeedback_{split}.jsonl")))

## UltraFeedback DPO (GPU 0 policy / GPU 1 reference) + HH-DPO generations (GPU 1)

In [ ]:
UF_OUT = OUT / "ultrafeedback_dpo"
bg = Background("hh_dpo_generations", [
    gen_cmd(PREV_HH_BEST, HH_EVAL, GEN / "hh_dpo_hh_eval.json", "qwen2.5-0.5b-hh-dpo", "hh_rlhf_style_eval"),
], gpus="1" if torch.cuda.device_count() > 1 else "0")
if done(UF_OUT):
    print("UltraFeedback DPO already done:", json.loads((UF_OUT / "summary.json").read_text()))
else:
    run(["python", "scripts/train_dpo.py", "--sft_checkpoint", PREV_SFT_BEST,
         "--train", UF_DIR / "ultrafeedback_train.jsonl", "--validation", UF_DIR / "ultrafeedback_valid.jsonl",
         "--output_dir", UF_OUT, "--run_name", "ultrafeedback_dpo", "--seed", SEED, *flags(DPO_HP), *SMOKE_DPO])
print(json.loads((UF_OUT / "summary.json").read_text()))
bg.wait()

In [ ]:
run(gen_cmd(UF_OUT / "best", ALPACA_EVAL, GEN / "ultrafeedback_dpo_alpaca_eval.json", "qwen2.5-0.5b-ultrafeedback-dpo",
            "alpaca_eval", ["--print_examples", 3]), gpus="0")
prune_weights(UF_OUT / "best", UF_OUT / "final")
for name in ("hh_dpo_hh_eval.json", "ultrafeedback_dpo_alpaca_eval.json"):
    rows = json.loads((GEN / name).read_text())
    print(f"{name}: {len(rows)} outputs, {sum(not r['output'].strip() for r in rows)} empty")

## Download bundle (merge with the first run's outputs afterwards)

In [ ]:
WORK = Path("/kaggle/working")
if OUTPUT_MODE == "lean":
    for path in DATA.rglob("*.jsonl"):
        path.unlink()  # copies of the first run's splits; metadata stays
bundle = WORK / ("continuation_bundle_smoke.zip" if SMOKE_TEST else "continuation_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT.rglob("*")):
        rel = path.relative_to(OUT)
        if path.is_file() and path.suffix != ".safetensors":
            zf.write(path, Path("outputs") / rel)
print(f"{bundle}  {bundle.stat().st_size / 1e6:.1f} MB")
(OUT / "PREVIOUS_RUN.txt").write_text(f"Continuation of the first run. SFT/HH-DPO checkpoints read from: {PREV}\n")
os.chdir(WORK)
display(FileLink(bundle.name))
os.chdir(repo)
subprocess.run(["du", "-sh", str(OUT)], check=False)
print("Done. Save Version, then merge continuation_bundle.zip with the first run's outputs.")